In [41]:
import pandas as pd

In [42]:
df_query = pd.read_csv('input/imb_raw_questions_20261002.csv',na_filter=False)
len(df_query)

2037

In [43]:
df_query

,model_name,question_package,field_name,max_value,count_of_values,case_counts
0,DiseaseSurveillanceModel_General,CLINICAL,ACF_NO,NO,210,17608
1,DiseaseSurveillanceModel_General,CLINICAL,ACF_UNKNOWN,UNKNOWN,26,17608
2,DiseaseSurveillanceModel_General,CLINICAL,ACF_YES,YES,164,17608
3,DiseaseSurveillanceModel_Hep,RISK_HISTORY,ACUPUNTURE_NO,NO,107,55428
4,DiseaseSurveillanceModel_Hep,RISK_HISTORY,ACUPUNTURE_UNKNOWN,UNKNOWN,63,55428
...,...,...,...,...,...,...
2032,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_SITE_NAME_STREET2,#100,1,17608
2033,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_WITHIN_LAST_6_MONTHS_NO,NO,66,17608
2034,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_WITHIN_LAST_6_MONTHS_YES,YES,1,17608
2035,DiseaseSurveillanceModel_Hep,DEMOGRAPHIC,ZIP,99999,53397,55428


In [44]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:].lower()==answer.lower()):
        return question[:start].strip('_')
    else:
        return question

In [45]:
df_query.loc[:,'parent_question_id']  = [ join_all_except_last(q,a) for q,a in zip(df_query.loc[:,'field_name'],df_query.loc[:,'max_value'])]

In [46]:
df_query

,model_name,question_package,field_name,max_value,count_of_values,case_counts,parent_question_id
0,DiseaseSurveillanceModel_General,CLINICAL,ACF_NO,NO,210,17608,ACF
1,DiseaseSurveillanceModel_General,CLINICAL,ACF_UNKNOWN,UNKNOWN,26,17608,ACF
2,DiseaseSurveillanceModel_General,CLINICAL,ACF_YES,YES,164,17608,ACF
3,DiseaseSurveillanceModel_Hep,RISK_HISTORY,ACUPUNTURE_NO,NO,107,55428,ACUPUNTURE
4,DiseaseSurveillanceModel_Hep,RISK_HISTORY,ACUPUNTURE_UNKNOWN,UNKNOWN,63,55428,ACUPUNTURE
...,...,...,...,...,...,...,...
2032,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_SITE_NAME_STREET2,#100,1,17608,WORK_SITE_NAME_STREET2
2033,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_WITHIN_LAST_6_MONTHS_NO,NO,66,17608,WORK_WITHIN_LAST_6_MONTHS
2034,DiseaseSurveillanceModel_General,DEMOGRAPHIC,WORK_WITHIN_LAST_6_MONTHS_YES,YES,1,17608,WORK_WITHIN_LAST_6_MONTHS
2035,DiseaseSurveillanceModel_Hep,DEMOGRAPHIC,ZIP,99999,53397,55428,ZIP


In [47]:
# Group by parent_question_id and model_name to get the num of values reported
df_group = df_query.groupby(by=['parent_question_id','model_name','question_package','case_counts'])['count_of_values'].sum().reset_index(name='count_of_values')

In [48]:
df_group.head()

,parent_question_id,model_name,question_package,case_counts,count_of_values
0,,DiseaseSurveillanceModel_General,ADMINISTRATIVE,17608,2
1,ACF,DiseaseSurveillanceModel_General,CLINICAL,17608,400
2,ACUPUNTURE,DiseaseSurveillanceModel_Hep,RISK_HISTORY,55428,172
3,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_General,ADMINISTRATIVE,17608,17037
4,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,55428,55533


In [49]:
df_group.loc[:,'fill_rate'] = [round(c*100.0/t,3) for c,t in zip(df_group.loc[:,'count_of_values'],df_group.loc[:,'case_counts'])]

In [50]:
df_group 

,parent_question_id,model_name,question_package,case_counts,count_of_values,fill_rate
0,,DiseaseSurveillanceModel_General,ADMINISTRATIVE,17608,2,0.011
1,ACF,DiseaseSurveillanceModel_General,CLINICAL,17608,400,2.272
2,ACUPUNTURE,DiseaseSurveillanceModel_Hep,RISK_HISTORY,55428,172,0.310
3,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_General,ADMINISTRATIVE,17608,17037,96.757
4,ADDITIONAL_SOURCE_CASE_REPORT,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,55428,55533,100.189
...,...,...,...,...,...,...
1329,WORK_SITE_NAME_STREET1,DiseaseSurveillanceModel_General,DEMOGRAPHIC,17608,56,0.318
1330,WORK_SITE_NAME_STREET2,DiseaseSurveillanceModel_General,DEMOGRAPHIC,17608,1,0.006
1331,WORK_WITHIN_LAST_6_MONTHS,DiseaseSurveillanceModel_General,DEMOGRAPHIC,17608,67,0.381
1332,ZIP,DiseaseSurveillanceModel_General,DEMOGRAPHIC,17608,15981,90.760


In [51]:
df_group = df_group[['model_name','question_package','parent_question_id','case_counts','count_of_values','fill_rate']]

In [52]:
df_group.to_csv('output/fill_rate_IMB_fields_20261002.csv',index=False)